# 2 $\sigma(Wx + b)$ and $\sigma(W * x)$

Notebook 01 handed you 384 numbers per image and left the obvious question open: **where
did those numbers come from?**

They came from about a dozen stacked applications of the two lines this notebook is about.
A layer is a linear operator, a bias, and a nonlinearity — and that is true whether the
linear operator is a dense matrix or a convolution. Courses usually teach those as two
different animals, in two different chapters. They are the same construction, and teaching
them together is both shorter and more honest.

The part that is *not* optional is the nonlinearity. That is what this notebook is really
about, and §2.1 and §2.2 exist to show you why.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn

from notebooktester import param

torch.manual_seed(0)

## 2.1 A stack of linear layers is one linear layer

Five `nn.Linear` layers, no activations between them. It looks like a deep network. Ask
what it can compute.

Each layer does $x \mapsto W_i x$. Applying five of them is $W_5 W_4 W_3 W_2 W_1 x$ — and
the product of five matrices is *a matrix*. So there is a single `nn.Linear` somewhere that
does exactly the same thing. We can build it, by multiplying the weights out by hand.

In [ ]:
WIDTH = 32
layers = [nn.Linear(WIDTH, WIDTH, bias=False) for _ in range(5)]
stack = nn.Sequential(*layers)

# collapse the five weight matrices into one.
# forward order is layers[0] -> ... -> layers[4], so the matrix product runs backwards
W = layers[-1].weight
for layer in reversed(layers[:-1]):
    W = W @ layer.weight

x = torch.rand(8, WIDTH)

with torch.no_grad():
    deep, flat = stack(x), x @ W.T

print("five layers :", tuple(deep.shape), " parameters:", sum(p.numel() for p in stack.parameters()))
print("one matrix  :", tuple(flat.shape), " parameters:", W.numel())
print()
print(f"max abs difference: {(deep - flat).abs().max():.2e}")
assert torch.allclose(deep, flat, atol=1e-5)
print("identical ✓")

5120 parameters doing the work of 1024. Not *approximately* the same function — the same
function, to floating-point precision.

This is worth being blunt about: **without a nonlinearity, depth buys you exactly
nothing.** Not "less than you hoped". Nothing. Every extra layer is a slower way to compute
a linear map you could have written down as one matrix.

(Biases change nothing here. $W_2(W_1x + b_1) + b_2 = (W_2W_1)x + (W_2b_1 + b_2)$ — still
one matrix and one bias. Try it with `bias=True` if you want to see it.)

## 2.2 The same is true of convolutions

A convolution is a linear operator too. It has two constraints a dense layer does not:

- **weight sharing** — the same small kernel is applied at every position, instead of a
  fresh weight per input-output pair;
- **locality** — an output only depends on a small window of the input.

Those constraints are what make it *useful* on images: they encode the assumption that a
pattern means the same thing wherever it appears, and they cut the parameter count by
orders of magnitude. But they constrain *which* linear maps the layer can express. They do
not make it a different kind of operation.

The test is the same one we just ran. If a conv layer is linear, two stacked convs with no
activation between them should collapse into a single conv — a wider one, since the
receptive fields compose.

In [ ]:
signal = torch.rand(1, 1, 20)                   # (B, C, L) -- channels second, see tensor-ladder.md

c1 = nn.Conv1d(1, 1, kernel_size=3, bias=False)
c2 = nn.Conv1d(1, 1, kernel_size=3, bias=False)

# the composed kernel: every way of picking one tap from each kernel
w1, w2 = c1.weight.flatten(), c2.weight.flatten()
composed = torch.zeros(len(w1) + len(w2) - 1)
for i, a in enumerate(w1):
    for j, b in enumerate(w2):
        composed[i + j] += a * b

single = nn.Conv1d(1, 1, kernel_size=5, bias=False)
with torch.no_grad():
    single.weight.copy_(composed.view(1, 1, -1))

with torch.no_grad():
    two, one = c2(c1(signal)), single(signal)

print("two convs, k=3 each :", tuple(two.shape), " kernel taps:", w1.numel() + w2.numel())
print("one conv,  k=5      :", tuple(one.shape), " kernel taps:", composed.numel())
print()
print(f"max abs difference: {(two - one).abs().max():.2e}")
assert torch.allclose(two, one, atol=1e-6)
print("identical ✓")

Same story, same verdict. Two 3-taps become one 5-tap: the receptive field grew, the
expressible function did not.

Note what the collapse *did* change — the shape. Each `kernel_size=3` with no padding costs
two positions, so 20 → 18 → 16, and the single 5-tap goes 20 → 16 directly. Stride and
padding change these numbers; they do not change the argument. Predicting output shapes
before running is exercise E1.

So we have two flavours of the same thing:

| | linear operator | constraint | shape it wants |
|---|---|---|---|
| **dense** | $Wx$, a full matrix | none — every input touches every output | `(..., D)` |
| **conv** | $W * x$, a shared kernel | weight sharing + locality | `(B, C, L)` or `(B, C, H, W)` |

and neither of them can express anything a single layer could not, no matter how many you
stack. Which brings us to the ingredient that changes that.

In [ ]:
# one picture, nothing more: which of the 20 input positions feed a single output
# position of the composed 5-tap kernel (output index 5 -> inputs 5..9)
receptive_field = range(5, 10)
colors = ["tab:orange" if i in receptive_field else "lightgray" for i in range(20)]

fig, ax = plt.subplots(figsize=(9, 1))
ax.bar(range(20), [1] * 20, color=colors, edgecolor="white", width=1.0)
ax.set_yticks([])
ax.set_xticks(range(20))
ax.set_title("one output position of the 5-tap kernel depends on these 5 input positions")
plt.tight_layout()
plt.show()

## 2.3 What the nonlinearity is for

Put a `GELU` between the layers of §2.1 and re-run the collapse. The matrix product is
still a perfectly good matrix — it just no longer describes what the network does.

In [ ]:
gelu_stack = nn.Sequential(
    layers[0], nn.GELU(),
    layers[1], nn.GELU(),
    layers[2], nn.GELU(),
    layers[3], nn.GELU(),
    layers[4],
)

x = torch.rand(8, WIDTH)
with torch.no_grad():
    bent, flat = gelu_stack(x), x @ W.T

print(f"max abs difference from the collapsed matrix: {(bent - flat).abs().max():.4f}")
print("still identical?", torch.allclose(bent, flat, atol=1e-5))

That was a numeric argument: two functions, compared by how close their outputs land.
Here is a visual one, on a task simple enough to see completely.

Four clusters sit at the corners of a square, coloured so that **diagonal corners share a
class** — this is XOR, the textbook example of "not linearly separable," old but exactly
the point. No single straight line divides the plane into "class 0" and "class 1"
correctly, whatever its angle or offset. Watch what a `Linear` layer can do with that
compared to a GELU network.

In [ ]:
torch.manual_seed(0)
n = 300
corners = torch.randint(0, 2, (n, 2)).float() * 2 - 1        # each coord is +-1
points = corners + 0.25 * torch.randn(n, 2)                  # a cloud around each corner
xor_labels = (corners[:, 0] * corners[:, 1] < 0).long()       # opposite-sign corners -> class 1


def fit_classifier(model, steps=param(400, test=20), lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        opt.zero_grad()
        loss_fn(model(points), xor_labels).backward()
        opt.step()
    return model


torch.manual_seed(0)
linear_clf = fit_classifier(nn.Linear(2, 2))
torch.manual_seed(0)
nonlin_clf = fit_classifier(nn.Sequential(nn.Linear(2, 32), nn.GELU(), nn.Linear(32, 2)))

gx, gy = torch.meshgrid(torch.linspace(-1.8, 1.8, 150), torch.linspace(-1.8, 1.8, 150), indexing="xy")
grid = torch.stack([gx.flatten(), gy.flatten()], dim=1)
colors = ["tab:blue" if label == 0 else "tab:orange" for label in xor_labels]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, model, title in [(axes[0], linear_clf, "linear: Wx + b"), (axes[1], nonlin_clf, "nonlinear: GELU MLP")]:
    with torch.no_grad():
        boundary = model(grid).argmax(1).reshape(gx.shape)
        accuracy = (model(points).argmax(1) == xor_labels).float().mean()
    ax.contourf(gx, gy, boundary, levels=[-0.5, 0.5, 1.5], alpha=0.3, colors=["tab:blue", "tab:orange"])
    ax.scatter(points[:, 0], points[:, 1], c=colors, edgecolor="k", s=20)
    ax.set_title(f"{title}\naccuracy {accuracy:.2f}")
plt.tight_layout()
plt.show()

The linear model's boundary is a straight line (or close to it — §2.1 proved that is all a
stack of `Linear`s can ever draw), and a straight line gets roughly half of XOR right
regardless of how long you train it. That is not a training failure, it is a
representational one: no setting of `W` and `b` fixes it. The GELU model bends the boundary
around the diagonal split and gets it essentially right.

Same idea, now as a regression instead of a classification, and with a strong quantitative
claim instead of just a picture: the nonlinear stack can fit functions that **no** linear
map can fit, no matter how it is trained.

Here is one. $y = \sin(3x)$ on $[-3, 3]$: a linear model has two parameters to describe it
with, and there is no choice of the two that works.

In [ ]:
STEPS = param(2000, test=50)

xs = torch.linspace(-3, 3, 200).unsqueeze(1)
true_ys = torch.sin(3 * xs)

torch.manual_seed(1)
ys = true_ys + 0.15 * torch.randn_like(true_ys)   # noisy targets -- something to actually fit


def fit(model, steps=STEPS, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = ((model(xs) - ys) ** 2).mean()
        loss.backward()
        opt.step()
    return model, float(loss.detach())


torch.manual_seed(0)
linear, linear_loss = fit(nn.Sequential(nn.Linear(1, 64), nn.Linear(64, 64), nn.Linear(64, 1)))
torch.manual_seed(0)
nonlin, nonlin_loss = fit(nn.Sequential(nn.Linear(1, 64), nn.GELU(), nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 1)))

print(f"three linear layers, no activation : final MSE {linear_loss:.3f} (against noisy targets)")
print(f"the same three, with GELU          : final MSE {nonlin_loss:.3f} (against noisy targets)")
print(f"\nthe nonlinear stack is {linear_loss / max(nonlin_loss, 1e-12):.1f}x better on the same data")

In [ ]:
with torch.no_grad():
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.scatter(xs, ys, color="gray", s=8, alpha=0.5, label="noisy training targets")
    ax.plot(xs, true_ys, color="k", lw=2, label="true function: sin(3x)")
    ax.plot(xs, linear(xs), lw=1.5, ls="--", label=f"linear stack (MSE {linear_loss:.3f})")
    ax.plot(xs, nonlin(xs), lw=1.5, label=f"with GELU (MSE {nonlin_loss:.3f})")
    ax.legend()
    ax.set_title("the same three layers, with and without the one ingredient")
    plt.tight_layout()

The linear stack draws the best straight line through a sine wave, because a straight line
is the only thing it has. Adding depth would not help — §2.1 already proved the extra layers
collapse away. The *only* edit that changes what is reachable is the activation.

### Which activation?

Less consequential than the fact that there is one, but the differences are real:

In [ ]:
z = torch.linspace(-4, 4, 200)
acts = {"sigmoid": torch.sigmoid, "tanh": torch.tanh, "ReLU": torch.relu, "GELU": torch.nn.functional.gelu}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for name, fn in acts.items():
    axes[0].plot(z, fn(z), lw=1.5, label=name)
    slope = (fn(z)[1:] - fn(z)[:-1]) / (z[1] - z[0])      # finite difference is plenty here
    axes[1].plot(z[:-1], slope, lw=1.5, label=name)
axes[0].set_title("activation")
axes[1].set_title("its slope -- what the gradient gets multiplied by")
axes[0].legend(fontsize=8)
axes[0].axhline(0, color="k", lw=0.5)
axes[1].axhline(0, color="k", lw=0.5)
plt.tight_layout()

Read the right-hand panel, because that is the one that decides whether training works.

- **sigmoid / tanh saturate.** Away from zero the slope goes to zero, so the gradient
  arriving from above gets multiplied by almost nothing. Stack a few layers of that and the
  early layers stop learning — the *vanishing gradient* problem. This is the historical
  reason deep nets were considered untrainable.
- **ReLU** has slope exactly 1 on the positive side, which fixed that. Its failure mode is
  the other one: a unit pushed permanently negative has slope 0 forever and never
  recovers — a *dead* ReLU.
- **GELU** is ReLU with the corner smoothed off, which keeps a small gradient alive on the
  negative side. It is the default in modern transformers, and it is what
  `vectormesh.components.NeuralNet` uses.

Pick GELU or ReLU and move on. The choice between them is worth far less of your attention
than the choice of encoder in notebook 01 — and notebook 04 will put a number on that claim.

## 2.4 Reading the encoder you have been using

Close the loop. Those 384 numbers in your cache came out of a stack of exactly these two
lines. Count what is actually inside the two encoders from notebook 01.

In [ ]:
from collections import Counter

from transformers import AutoModel

for tag in ("microsoft/resnet-18", "facebook/dinov2-small"):
    model = AutoModel.from_pretrained(tag)
    counts = Counter(type(m).__name__ for _, m in model.named_modules())
    params = sum(p.numel() for p in model.parameters())

    print(f"\n{tag}   ({params / 1e6:.1f}M parameters)")
    print("  top-level:", [name for name, _ in model.named_children()])
    for kind in ("Conv2d", "Linear", "BatchNorm2d", "LayerNorm", "ReLU", "GELUActivation"):
        if counts[kind]:
            print(f"    {counts[kind]:3d}  {kind}")

Two architectures, one recipe.

**resnet-18** is 20 `Conv2d`, 20 `BatchNorm2d`, 17 `ReLU`. That is §2.2's linear operator,
a normalisation, and §2.3's nonlinearity — repeated seventeen times.

**dinov2-small** is 72 `Linear`, 25 `LayerNorm`, 12 `GELUActivation`. Same recipe, dense
instead of convolutional, twelve times. (The `Linear` count is high because each attention
block spends four of them on queries, keys, values and the output projection — lesson 5's
subject, not this one.)

And look at the top of the DINOv2 count: **one `Conv2d`**. A vision transformer is not
conv-free. It uses exactly one convolution, at the very start, with a stride equal to its
kernel size — which is a tidy way of saying "cut the image into non-overlapping patches and
project each one". §2.2's operator, used once, as a tokenizer.

Neither of them is doing anything you have not now built by hand. The difference between
your five-layer stack and DINOv2 is not the construction; it is 22 million parameters and
a very large pile of images, and *that* is the thing you downloaded rather than trained.

Which is the point notebook 01 made with a stopwatch, arrived at from the other direction:
the encoder is the expensive, valuable part, and it is made of this.

## What you now know

- A stack of linear layers, dense or convolutional, collapses into a single layer. Depth
  without a nonlinearity is a slower way to compute the same function.
- A convolution is a linear operator with weight sharing and locality. Those constraints
  buy efficiency and an assumption about the data; they do not change what kind of
  operation it is.
- The activation is the one ingredient that makes depth mean something.
- Saturation and dead units are the two failure modes; GELU/ReLU avoid the first, GELU
  softens the second.
- Your cached encoder is this construction, repeated a dozen times, with weights someone
  else paid to fit.

## Exercises

**E1 — predict the shape.** For `x` of shape `(1, 1, 20)`, write down the output shape of
`nn.Conv1d(1, 1, kernel_size=5)` with (a) `stride=1, padding=0`, (b) `stride=2, padding=2`.
Then run it. If you were wrong, work out which term in $\lfloor (L + 2p - k)/s \rfloor + 1$
you mis-read.

**E2 — kill the nonlinearity in something that works.** Take the sine fit from §2.3 and
replace `nn.GELU()` with `nn.Identity()`, leaving everything else untouched. Predict the
final MSE before running it. You already know enough to predict it exactly — say which
number, and why.

**E3 — how deep before it stops mattering?** Repeat §2.1 with 2, 5, 10 and 50 layers and
time the forward pass of the stack against the collapsed matrix. Depth costs you something
even when it buys nothing; how much?

**E4 — the honest version of the claim.** §2.1 shows the collapse for `bias=False`. Redo it
with `bias=True`: find the single weight matrix *and* the single bias vector that reproduce
the five-layer stack, and assert it.